## 212. Word Search II
- Description:
  <blockquote>
  [212. Word Search II](https://leetcode.com/problems/word-search-ii/) Given an `m x n` `board` of characters and a list of strings `words`, return  *all words on the board* .
 
Each word must be constructed from letters of sequentially adjacent cells, where **adjacent cells** are horizontally or vertically neighboring. The same letter cell may not be used more than once in a word.
 
**Example 1:**
![Image](https://assets.leetcode.com/uploads/2020/11/07/search1.jpg)
 
**Input:** board = `["o", "a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]`, words = ["oath","pea","eat","rain"]
**Output:** ["eat","oath"]
 
**Example 2:**
![Image](https://assets.leetcode.com/uploads/2020/11/07/search2.jpg)
 
**Input:** board = `["a", "b"],["c","d"]`, words = ["abcb"]
**Output:** []
 
**Constraints:**
 
- `m == board.length`
- `n == board[i].length`
- `1 <= m, n <= 12`
- `board[i][j]` is a lowercase English letter.
- `1 <= words.length <= 3 * 104`
- `1 <= words[i].length <= 10`
- `words[i]` consists of lowercase English letters.
- All the strings of `words` are unique.
  </blockquote>

- URL: [Problem_URL](https://leetcode.com/problems/word-search-ii/description/)

- Topics: Backtracking, Trie

- Difficulty: Hard

- Resources: example_resource_URL

### Solution 1, Trie-Guided Backtracking with Incremental Leaf Pruning optimization

W = number of words, 
L = max word length
M = No of Rows
N = No of Cols
C = Total No of Cells AKA M*N

- Time Complexity: O(W·L + M·N·4·3^(L-1)) ~ O(W*L + C*4*3^(L-1))
1. Trie Construction → O(W · L)

    W = number of words, L = max word length
    Each word is inserted character by character

2. Backtracking → O(M · N · 4 · 3^(L-1))

    M·N = every cell is a potential starting point
    From the first cell: 4 possible directions
    From each subsequent cell: at most 3 directions (one cell is marked #, eliminating going back)
    Max depth = L (max word length — trie limits how deep we go)

A word of length L requires L steps on the board. The first first step has 4 possible directions we can go in, the next step onwards you only have 3 directions as the previous cell we moved from is marked as '#' to prevent going back to it, so every step after the first will have 3 directions and we will take L-1 such steps for a word of length L.

- Space Complexity: O(W · L)
Trie nodes - O(W · L)
Recursion call stack - O(L) — depth bounded by max word length
matchedWords output - O(W)

Trie dominates → O(W · L)

Worth noting: the leaf pruning optimization progressively shrinks the trie during search, improving practical time and space performance beyond the theoretical worst case, though it doesn't change the asymptotic bounds.
Solution description

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}  # replaces the nested dict
        self.word = None    # replaces WORD_KEY "$"

class Solution:
    def findWords(self, board: List[List[str]], words: List[str]) -> List[str]:

        # Build the Trie
        root = TrieNode()
        for word in words:
            node = root
            for letter in word:
                if letter not in node.children:
                    node.children[letter] = TrieNode()  # replaces setdefault()
                node = node.children[letter]
            # we store the entire word to keep the backtracking function simpler and cleaner, to avoid string concatenation at every step to build the word during traversal
            node.word = word

        rowSize = len(board)
        colSize = len(board[0])
        matchedWords = []

        def backtracking(row, col, parent):
            letter = board[row][col]
            currNode = parent.children[letter]

            if currNode.word:
                matchedWords.append(currNode.word)
                # reset to avoid duplicate words, aka once we match a word and add it to matchedWords we do not want that word to be matched again through a different path on the board
                currNode.word = None
                # we continue with the backtracking to find any longer words extending from here

            board[row][col] = "#"

            for rowOffset, colOffset in [(-1, 0), (0, 1), (1, 0), (0, -1)]:
                newRow, newCol = row + rowOffset, col + colOffset
                if newRow < 0 or newRow >= rowSize or newCol < 0 or newCol >= colSize:
                    continue
                if board[newRow][newCol] not in currNode.children:
                    continue
                backtracking(newRow, newCol, currNode)

            board[row][col] = letter

            # Optimization - Gradually prune the nodes in Trie during the backtracking
            # For a leaf node in Trie, once we traverse it (i.e. find a matched word), we would no longer need to traverse it again. As a result, we could prune it out from the Trie.
            # we only prune when currNode.children is empty, meaning truly no further words can extend from that node.
            if not currNode.children:
                parent.children.pop(letter)

        # Iterate through each cell of the board
        for row in range(rowSize):
            for col in range(colSize):
                # Only proceed if the current cell letter is in the Trie, aka part of the words list
                if board[row][col] in root.children:
                    backtracking(row, col, root)

        return matchedWords